# 01 — Data Exploration
Load the raw NIfTI scan and inspect its geometry, metadata, and intensity distribution.
We'll visualize the middle slice of all three anatomical planes to confirm the data is intact.

In [ ]:
import nibabel as nib
import numpy as np
import matplotlib.pyplot as plt
import os

## 1. Load the NIfTI file

In [ ]:
DATA_PATH = os.path.join("..", "data", "raw", "brain_scan.nii.gz")

img = nib.load(DATA_PATH)
data = img.get_fdata()          # loads the 3-D array as float64

print("Shape  :", data.shape)   # expected: (X, Y, Z) voxels
print("Dtype  :", data.dtype)
print("Affine :\n", img.affine)  # voxel → mm mapping
print("Header :", img.header.get_data_shape(), "|",
      "voxel size:", img.header.get_zooms())

## 2. Intensity statistics

In [ ]:
print(f"Min  : {data.min():.2f}")
print(f"Max  : {data.max():.2f}")
print(f"Mean : {data.mean():.2f}")
print(f"Std  : {data.std():.2f}")

fig, ax = plt.subplots(figsize=(8, 3))
ax.hist(data[data > 0].ravel(), bins=200, color="steelblue", edgecolor="none")
ax.set_title("Voxel Intensity Histogram (non-zero voxels)")
ax.set_xlabel("Intensity")
ax.set_ylabel("Count")
plt.tight_layout()
plt.savefig(os.path.join("..", "outputs", "figures", "01_intensity_histogram.png"), dpi=150)
plt.show()

## 3. Middle-slice visualization — all three anatomical planes

NIfTI axes follow the RAS+ convention:
- **Axis 0** → left/right → sagittal slices  
- **Axis 1** → posterior/anterior → coronal slices  
- **Axis 2** → inferior/superior → axial slices  

In [ ]:
def plot_middle_slices(volume, title="Middle Slices", save_path=None):
    """
    Plot the center slice of a 3-D volume in axial, coronal, and sagittal planes.

    Parameters
    ----------
    volume    : np.ndarray  shape (X, Y, Z)
    title     : str
    save_path : str | None  — if provided, saves the figure to disk
    """
    x_mid = volume.shape[0] // 2
    y_mid = volume.shape[1] // 2
    z_mid = volume.shape[2] // 2

    slices = {
        "Sagittal  (axis 0)": volume[x_mid, :, :],
        "Coronal   (axis 1)": volume[:, y_mid, :],
        "Axial     (axis 2)": volume[:, :, z_mid],
    }

    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    fig.suptitle(title, fontsize=14, fontweight="bold")

    for ax, (label, sl) in zip(axes, slices.items()):
        # Rotate so superior is up, anterior is right (standard radiological view)
        ax.imshow(np.rot90(sl), cmap="gray", origin="lower")
        ax.set_title(label)
        ax.axis("off")

    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches="tight")
        print(f"Saved → {save_path}")
    plt.show()


plot_middle_slices(
    data,
    title="brain_scan.nii.gz — Middle Slices (raw)",
    save_path=os.path.join("..", "outputs", "figures", "01_middle_slices.png"),
)

## 4. Quick sanity checks
Confirm the scan isn't corrupted or oddly oriented before we hand it to MONAI.

In [ ]:
# Check for NaNs / Infs
print("NaN voxels :", np.isnan(data).sum())
print("Inf voxels :", np.isinf(data).sum())

# Confirm orientation code from the affine
try:
    orientation = nib.aff2axcodes(img.affine)
    print("Orientation:", orientation)   # ideally ('R','A','S') or similar RAS+ variant
except Exception as e:
    print("Could not determine orientation:", e)

print("\nAll checks passed — ready for preprocessing.")